# Linear Maps and Matrices

Concept `06-linear-maps` of the math-foundations learning map.

A **linear map** is a function $T : V \to W$ between vector spaces that
preserves vector addition and scalar multiplication. Once bases are fixed,
$T$ is encoded by a matrix. This notebook walks through the definition,
the rank-nullity theorem, a worked example (rotation in the plane), and
the composition-equals-matrix-product identity.


## Definitions

* **Linear map.** $T(\alpha u + \beta v) = \alpha T(u) + \beta T(v)$.
* **Kernel.** $\ker T = \{ v \in V : T(v) = 0 \}$, a subspace of $V$.
* **Image.** $\operatorname{im} T = \{ T(v) : v \in V \}$, a subspace of $W$.
* **Rank, nullity.** $\operatorname{rank} T = \dim \operatorname{im} T$,
  $\operatorname{nullity} T = \dim \ker T$.
* **Matrix.** With bases $(v_j)$ for $V$, $(w_i)$ for $W$, write
  $T(v_j) = \sum_i a_{ij} w_i$; the matrix is $[T] = (a_{ij})$.


In [ ]:
import numpy as np

def null_space(A, tol=1e-9):
    _, s, vt = np.linalg.svd(A)
    n = A.shape[1]
    s_full = np.zeros(n); s_full[:len(s)] = s
    return vt[s_full <= tol].T

def column_space(A, tol=1e-9):
    u, s, _ = np.linalg.svd(A, full_matrices=False)
    r = int(np.sum(s > tol))
    return u[:, :r]

A = np.array([[1., 2., 3.],
              [1., 2., 3.],
              [4., 5., 6.]])

K = null_space(A)
C = column_space(A)
rank = int(np.linalg.matrix_rank(A))
nullity = K.shape[1]

print('A =\n', A)
print('rank   =', rank)
print('nullity=', nullity)
print('rank + nullity =', rank + nullity, '== dim(domain) =', A.shape[1])

# Sanity: A @ (kernel basis) ~ 0
print('max |A @ K| =', float(np.max(np.abs(A @ K))))


## Worked Example: Rotation in $\mathbb{R}^2$

The counter-clockwise rotation $R_\theta$ is linear and, in the standard
basis, has matrix
$$
[R_\theta] = \begin{pmatrix} \cos\theta & -\sin\theta \\
\sin\theta & \cos\theta \end{pmatrix}.
$$
Its kernel is $\{0\}$ and its image is all of $\mathbb{R}^2$, so
rank-nullity reads $0 + 2 = 2$. Composing two rotations adds the angles,
which is just the matrix identity $[R_\alpha][R_\beta] = [R_{\alpha+\beta}]$.


In [ ]:
def R(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

alpha, beta = 0.3, 1.1
lhs = R(alpha) @ R(beta)
rhs = R(alpha + beta)
print('||R(a)R(b) - R(a+b)||_F =', float(np.linalg.norm(lhs - rhs)))

# Rotation is injective: kernel is trivial.
print('rank R(theta)   =', int(np.linalg.matrix_rank(R(0.7))))
print('nullity R(theta)=', 2 - int(np.linalg.matrix_rank(R(0.7))))


## Composition of Linear Maps = Matrix Product

For $S : U \to V$ and $T : V \to W$ with fixed bases,
$[T \circ S] = [T] \, [S]$. We check this numerically on random
$3 \times 3$ matrices.


In [ ]:
rng = np.random.default_rng(0)
S = rng.standard_normal((3, 3))
T = rng.standard_normal((3, 3))
composed = T @ S

X = rng.standard_normal((3, 5))
lhs = T @ (S @ X)
rhs = composed @ X
print('max |T(S(x)) - (TS)x| =', float(np.max(np.abs(lhs - rhs))))


## Exercises

1. Let $T(x,y,z) = (x+y, y+z)$ from $\mathbb{R}^3$ to $\mathbb{R}^2$.
   Compute $\ker T$ and $\operatorname{im} T$ by hand, then verify
   numerically using the helpers above.
2. Build the matrix of differentiation $D$ on polynomials of degree
   $\leq 3$ in the basis $(1, x, x^2, x^3)$; verify rank-nullity.
3. Sample two random $5 \times 5$ matrices $A, B$ with `rank(A) = 3` and
   `rank(B) = 2` (e.g. by taking truncated SVDs). Check empirically that
   $\operatorname{rank}(AB) \leq \min(\operatorname{rank} A,
   \operatorname{rank} B)$. Why is this the structural reason a
   rank-$r$ LoRA update can only move the layer along an $r$-dim
   subspace?
